# 🧠 Lesson 10: Memory Match

Can you remember where the animals are hiding? Flip two cards at a time
and try to find matching pairs. This game is powered by a Python **list**
— a way to keep many things in order.

In [1]:
import sys, os
sys.path.append(os.path.abspath(os.path.join("..", "..")))
from helpers.kid_tools import celebrate, try_again, check_answer, quiz, badge


## Lists hold many things

A list keeps several values in a row, like a row of cards:

```python
animals = ["🐶", "🐱", "🐸"]
```

Our game makes a deck with **two of each** animal, then shuffles it so
the pairs are hidden in random spots:

```python
deck = ANIMALS * 2      # two of each
random.shuffle(deck)    # mix them up
```

Run the game and find all the pairs!

In [2]:
import ipywidgets as widgets
from IPython.display import display
import random

ANIMALS = ["🐶", "🐱", "🐸", "🦊", "🐼", "🦁"]

deck = ANIMALS * 2          # each animal twice = pairs
random.shuffle(deck)        # mix them up

matched = [False] * len(deck)
face_up = []                # indices of cards currently turned over
moves = 0
status = widgets.Output()

buttons = []


def show_status(message=""):
    with status:
        status.clear_output()
        pairs_found = sum(matched) // 2
        print(f"🧠 Pairs found: {pairs_found} of {len(ANIMALS)}    Tries: {moves}")
        if message:
            print(message)


def make_card_handler(i):
    def handler(_):
        global moves
        # If two non-matching cards are already showing, turn them back first.
        if len(face_up) == 2:
            for j in face_up:
                buttons[j].description = "❓"
            face_up.clear()

        if matched[i] or i in face_up:
            return

        buttons[i].description = deck[i]
        face_up.append(i)

        if len(face_up) == 2:
            moves += 1
            a, b = face_up
            if deck[a] == deck[b]:
                matched[a] = True
                matched[b] = True
                buttons[a].style.button_color = "#a5d6a7"
                buttons[b].style.button_color = "#a5d6a7"
                face_up.clear()
                if all(matched):
                    show_status()
                    celebrate(f"You found every pair in {moves} tries!")
                    return
        show_status()
    return handler


for i in range(len(deck)):
    card = widgets.Button(description="❓",
                          layout=widgets.Layout(width="60px", height="60px"))
    card.style.font_size = "24px"
    card.on_click(make_card_handler(i))
    buttons.append(card)

# arrange the cards in rows of 4
rows = []
for start in range(0, len(buttons), 4):
    rows.append(widgets.HBox(buttons[start:start + 4]))

show_status("Click two cards to find a matching pair!")
display(widgets.VBox(rows))
display(status)


Output()

## How a match is checked

When two cards are face up, the game compares them:

```python
if deck[a] == deck[b]:   # same animal?
    matched[a] = True    # remember this card is done
    matched[b] = True
```

`deck[a]` means "the card at position `a` in the list." Comparing them
with `==` (just like lesson 5!) is how the game knows you found a pair.

In [6]:
weather="windy"

if weather=="raining":
    print("Stay at home")
elif weather=="sunny":
    print("Go to the beach")
elif weather 1:
    print()
elif weather 2:
    print()
else:
    print("Go somehwere")

Go somehwere


## Make it your own

Back in the game cell, try:

* Change the animals: `ANIMALS = ["🍎", "🍌", "🍇", "🍓", "🍊", "🍉"]`
* Make it easier with fewer pairs: `ANIMALS = ["🐶", "🐱", "🐸"]`
* Make it a bigger challenge by adding more animals to the list

## Quick quiz

In [7]:
import ipywidgets as widgets
from IPython.display import display
import random
import time
import threading

# Expanded set of animals (12 pairs = 24 cards total)
ANIMALS = ["🐶", "🐱", "🐸", "🦊", "🐼", "🦁", "🐯", "🐨", "🐷", "🐵", "🦄", "🐙"]

deck = ANIMALS * 2
random.shuffle(deck)

matched = [False] * len(deck)
face_up = []
moves = 0
start_time = None
elapsed_time = 0
game_over = False

status = widgets.Output()
grid_container = widgets.VBox()
buttons = []

# Timer thread function to update the elapsed time live
def timer_loop():
    global elapsed_time
    while not game_over:
        if start_time is not None:
            elapsed_time = int(time.time() - start_time)
            show_status()
        time.sleep(1)

def show_status(message=""):
    with status:
        status.clear_output(wait=True)
        pairs_found = sum(matched) // 2
        print(f"🧠 Pairs: {pairs_found}/{len(ANIMALS)} | 🎯 Moves: {moves} | ⏱️ Time: {elapsed_time}s")
        if message:
            print(message)

def make_card_handler(i):
    def handler(_):
        global moves, start_time, game_over
        
        if game_over:
            return

        # Start the timer on the very first card flip
        if start_time is None:
            start_time = time.time()
            threading.Thread(target=timer_loop, daemon=True).start()

        # Flip non-matching card pair back down if player clicks a third card
        if len(face_up) == 2:
            for j in face_up:
                buttons[j].description = "❓"
            face_up.clear()

        if matched[i] or i in face_up:
            return

        buttons[i].description = deck[i]
        face_up.append(i)

        if len(face_up) == 2:
            moves += 1
            a, b = face_up
            if deck[a] == deck[b]:
                matched[a] = True
                matched[b] = True
                buttons[a].style.button_color = "#a5d6a7"
                buttons[b].style.button_color = "#a5d6a7"
                face_up.clear()

                if all(matched):
                    game_over = True
                    final_time = int(time.time() - start_time)
                    show_status(f"🎉 Victory! Cleared all 12 pairs in {moves} moves and {final_time} seconds!")
                    return
            show_status()
            
    return handler

# Create buttons for all 24 cards
for i in range(len(deck)):
    card = widgets.Button(
        description="❓",
        layout=widgets.Layout(width="60px", height="60px")
    )
    card.style.font_size = "24px"
    card.on_click(make_card_handler(i))
    buttons.append(card)

# Arrange the 24 cards into 4 rows of 6 cards each
rows = []
for start in range(0, len(buttons), 6):
    rows.append(widgets.HBox(buttons[start:start + 6]))

grid_container.children = rows

show_status("Click any card to start the timer!")
display(grid_container)
display(status)

Output()

In [ ]:
quiz(
    question="Why do we write `deck = ANIMALS * 2` before shuffling?",
    choices=[
        "To make the game twice as fast",
        "So every animal appears twice, creating pairs to match",
        "To double the score",
    ],
    correct_index=1,
    hint="A matching game needs two of each card.",
)


What a memory! You just used lists to build a real game.

In [ ]:
badge("Memory Master", emoji="🧠")